In [1]:
# Cell 1: Build NH3 fermionic Hamiltonian and Hermitian fermionic terms

import math
import pandas as pd

from openfermion.chem import MolecularData
from openfermion.ops import FermionOperator
from openfermion.transforms import get_fermion_operator, normal_ordered
from openfermion.utils import hermitian_conjugated
from openfermionpyscf import run_pyscf

pd.set_option("display.max_colwidth", None)


# ------------------------------------------------------------
# Ammonia / NH3 configuration
# ------------------------------------------------------------

NH_BOND_LENGTH = 1.012  # Angstrom, approximate N-H bond length
HNH_ANGLE_DEGREES = 106.7  # approximate H-N-H angle
BASIS = "sto-3g"
MULTIPLICITY = 1
CHARGE = 0

# Full NH3/STO-3G should give 16 qubits:
# N has 5 STO-3G spatial orbitals and each H has 1.
# 8 spatial orbitals * 2 spin orbitals = 16 qubits.
USE_ACTIVE_SPACE = False

# Optional active-space example:
# Freeze the N core-like spatial orbital and keep valence orbitals active.
# This usually reduces NH3/STO-3G from 16 qubits to 14 qubits.
OCCUPIED_INDICES = [0]
ACTIVE_INDICES = [1, 2, 3, 4, 5, 6, 7]

TERM_ABS_TOL = 1e-12
PRINT_FULL_HAMILTONIAN = True


def format_fermion_term(term):
    if term == ():
        return "I"

    pieces = []
    for orbital, action in term:
        if action == 1:
            pieces.append(f"a_{orbital}^dagger")
        else:
            pieces.append(f"a_{orbital}")
    return " ".join(pieces)


def sort_fermion_key(term):
    return (len(term), term)


def coeff_to_str(c, digits=8):
    c = complex(c)
    if abs(c.imag) < 1e-12:
        return f"{c.real:+.{digits}f}"
    return f"{c.real:+.{digits}f}{c.imag:+.{digits}f}j"


def operator_to_string(op, digits=8):
    pieces = []

    for term, coeff in sorted(op.terms.items(), key=lambda item: sort_fermion_key(item[0])):
        pieces.append(f"{coeff_to_str(coeff, digits)} {format_fermion_term(term)}")

    if len(pieces) == 0:
        return "0"

    return " + ".join(pieces)


def dagger_term_key(term):
    """
    Return the OpenFermion key for O^dagger, where O is one monomial.
    """
    O = FermionOperator(term, 1.0)
    O_dag = normal_ordered(hermitian_conjugated(O))
    O_dag.compress(abs_tol=TERM_ABS_TOL)

    if len(O_dag.terms) != 1:
        raise ValueError(f"Expected one dagger term, got: {O_dag}")

    return next(iter(O_dag.terms.keys()))


def make_hermitian_fermionic_terms(fermion_hamiltonian, tol=TERM_ABS_TOL):
    """
    Group raw monomials into Hermitian fermionic Hamiltonian terms.
    """
    used = set()
    hermitian_terms = []

    for term, coeff in fermion_hamiltonian.terms.items():
        if term in used:
            continue

        dag = dagger_term_key(term)

        if dag == term:
            T = FermionOperator(term, coeff)
            used.add(term)
        else:
            dag_coeff = fermion_hamiltonian.terms.get(dag, 0.0)

            T = FermionOperator(term, coeff)
            T += FermionOperator(dag, dag_coeff)

            used.add(term)
            used.add(dag)

        T = normal_ordered(T)
        T.compress(abs_tol=tol)
        hermitian_terms.append(T)

    return hermitian_terms


def infer_n_qubits_from_fermion_operator(op):
    """
    Infer the number of spin orbitals used by the FermionOperator.
    """
    max_orbital = -1

    for term in op.terms:
        for orbital, action in term:
            max_orbital = max(max_orbital, orbital)

    return max_orbital + 1


def build_nh3_geometry(
    nh_bond_length=NH_BOND_LENGTH,
    hnh_angle_degrees=HNH_ANGLE_DEGREES,
):
    """
    Build trigonal-pyramidal NH3 geometry.

    N is placed at the origin.
    The three H atoms are placed symmetrically around the z-axis.
    The geometry approximately preserves the H-N-H angle.
    """
    gamma = math.radians(hnh_angle_degrees)

    # For three equivalent H atoms separated by 120 degrees in azimuth:
    # cos(gamma) = 1.5*cos(alpha)^2 - 0.5
    cos_alpha_squared = (math.cos(gamma) + 0.5) / 1.5
    cos_alpha_squared = max(0.0, min(1.0, cos_alpha_squared))

    cos_alpha = math.sqrt(cos_alpha_squared)
    sin_alpha = math.sqrt(1.0 - cos_alpha_squared)

    radial = nh_bond_length * sin_alpha
    z = nh_bond_length * cos_alpha

    geometry = [
        ("N", (0.0, 0.0, 0.0)),
        ("H", (radial, 0.0, z)),
        ("H", (radial * math.cos(2.0 * math.pi / 3.0), radial * math.sin(2.0 * math.pi / 3.0), z)),
        ("H", (radial * math.cos(4.0 * math.pi / 3.0), radial * math.sin(4.0 * math.pi / 3.0), z)),
    ]

    return geometry


def build_nh3_fermionic_hamiltonian(
    nh_bond_length=NH_BOND_LENGTH,
    hnh_angle_degrees=HNH_ANGLE_DEGREES,
    basis=BASIS,
    multiplicity=MULTIPLICITY,
    charge=CHARGE,
    use_active_space=USE_ACTIVE_SPACE,
    occupied_indices=OCCUPIED_INDICES,
    active_indices=ACTIVE_INDICES,
):
    """
    Build an NH3 fermionic Hamiltonian using OpenFermion + PySCF.
    """
    geometry = build_nh3_geometry(
        nh_bond_length=nh_bond_length,
        hnh_angle_degrees=hnh_angle_degrees,
    )

    molecule = MolecularData(
        geometry=geometry,
        basis=basis,
        multiplicity=multiplicity,
        charge=charge,
        description=f"NH3_{nh_bond_length}_{hnh_angle_degrees}",
    )

    molecule = run_pyscf(
        molecule,
        run_scf=True,
        run_fci=False,
    )

    if use_active_space:
        molecular_hamiltonian = molecule.get_molecular_hamiltonian(
            occupied_indices=occupied_indices,
            active_indices=active_indices,
        )
    else:
        molecular_hamiltonian = molecule.get_molecular_hamiltonian()

    fermion_hamiltonian = get_fermion_operator(molecular_hamiltonian)
    fermion_hamiltonian = normal_ordered(fermion_hamiltonian)
    fermion_hamiltonian.compress(abs_tol=TERM_ABS_TOL)

    n_qubits = infer_n_qubits_from_fermion_operator(fermion_hamiltonian)

    return molecule, fermion_hamiltonian, n_qubits


# ------------------------------------------------------------
# Build NH3 fermionic Hamiltonian
# ------------------------------------------------------------

molecule, Hf, n_qubits = build_nh3_fermionic_hamiltonian()
hermitian_terms = make_hermitian_fermionic_terms(Hf)

print("Molecule: NH3 / Ammonia")
print("Basis:", BASIS)
print("N-H bond length [Angstrom]:", NH_BOND_LENGTH)
print("H-N-H angle [degrees]:", HNH_ANGLE_DEGREES)
print("Use active space:", USE_ACTIVE_SPACE)
if USE_ACTIVE_SPACE:
    print("Frozen occupied spatial orbitals:", OCCUPIED_INDICES)
    print("Active spatial orbitals:", ACTIVE_INDICES)
print("Full molecule electrons:", molecule.n_electrons)
print("Full molecule spatial orbitals:", molecule.n_orbitals)
print("Full molecule spin orbitals / qubits:", molecule.n_qubits)
print("Hamiltonian spin orbitals / qubits used:", n_qubits)
print("Number of raw OpenFermion monomial terms:", len(Hf.terms))
print("Number of Hermitian fermionic terms:", len(hermitian_terms))

if PRINT_FULL_HAMILTONIAN:
    print("\n=== Full fermionic Hamiltonian H_f ===")
    print(Hf)


# ------------------------------------------------------------
# Tables
# ------------------------------------------------------------

raw_rows = []

for idx, (term, coeff) in enumerate(
    sorted(Hf.terms.items(), key=lambda item: sort_fermion_key(item[0]))
):
    raw_rows.append(
        {
            "raw_index": idx,
            "coefficient": coeff_to_str(coeff),
            "monomial": format_fermion_term(term),
            "OpenFermion_key": term,
        }
    )

raw_df = pd.DataFrame(raw_rows)

print("\n=== Raw fermionic monomials c_alpha O_alpha ===")
display(raw_df)


hermitian_rows = []

for i, T in enumerate(hermitian_terms):
    hermitian_rows.append(
        {
            "vertex": f"T_{i}",
            "number_of_monomials": len(T.terms),
            "fermionic_term": operator_to_string(T),
        }
    )

hermitian_df = pd.DataFrame(hermitian_rows)

print("\n=== Hermitian fermionic terms T_i ===")
display(hermitian_df)

Molecule: NH3 / Ammonia
Basis: sto-3g
N-H bond length [Angstrom]: 1.012
H-N-H angle [degrees]: 106.7
Use active space: False
Full molecule electrons: 10
Full molecule spatial orbitals: 8
Full molecule spin orbitals / qubits: 16
Hamiltonian spin orbitals / qubits used: 16
Number of raw OpenFermion monomial terms: 3609
Number of Hermitian fermionic terms: 1873

=== Full fermionic Hamiltonian H_f ===
11.958585125793121 [] +
-25.746794211057978 [0^ 0] +
0.4426997355499583 [0^ 2] +
-0.17053777967897008 [0^ 8] +
0.3516613863768956 [0^ 10] +
-4.126649890771728 [1^ 0^ 1 0] +
-0.34329807968342274 [1^ 0^ 2 1] +
0.34329807968342274 [1^ 0^ 3 0] +
-0.045379131115623306 [1^ 0^ 3 2] +
-0.009371722102649502 [1^ 0^ 5 4] +
-0.009371722102649483 [1^ 0^ 7 6] +
0.1373379759644902 [1^ 0^ 8 1] +
-0.014915715763750656 [1^ 0^ 8 3] +
-0.1373379759644902 [1^ 0^ 9 0] +
0.014915715763750656 [1^ 0^ 9 2] +
-0.025540051308932266 [1^ 0^ 9 8] +
-0.28942498341134315 [1^ 0^ 10 1] +
0.04001216491528481 [1^ 0^ 10 3] +
-0.0

,raw_index,coefficient,monomial,OpenFermion_key
0,0,+11.95858513,I,()
1,1,-25.74679421,a_0^dagger a_0,"((0, 1), (0, 0))"
2,2,+0.44269974,a_0^dagger a_2,"((0, 1), (2, 0))"
3,3,-0.17053778,a_0^dagger a_8,"((0, 1), (8, 0))"
4,4,+0.35166139,a_0^dagger a_10,"((0, 1), (10, 0))"
...,...,...,...,...
3604,3604,-0.06870661,a_15^dagger a_14^dagger a_15 a_4,"((15, 1), (14, 1), (15, 0), (4, 0))"
3605,3605,+0.01188250,a_15^dagger a_14^dagger a_15 a_6,"((15, 1), (14, 1), (15, 0), (6, 0))"
3606,3606,-0.00450867,a_15^dagger a_14^dagger a_15 a_8,"((15, 1), (14, 1), (15, 0), (8, 0))"
3607,3607,+0.01709649,a_15^dagger a_14^dagger a_15 a_10,"((15, 1), (14, 1), (15, 0), (10, 0))"



=== Hermitian fermionic terms T_i ===


,vertex,number_of_monomials,fermionic_term
0,T_0,1,+11.95858513 I
1,T_1,1,-25.74679421 a_0^dagger a_0
2,T_2,2,+0.44269974 a_0^dagger a_2 + +0.44269974 a_2^dagger a_0
3,T_3,2,-0.17053778 a_0^dagger a_8 + -0.17053778 a_8^dagger a_0
4,T_4,2,+0.35166139 a_0^dagger a_10 + +0.35166139 a_10^dagger a_0
...,...,...,...
1868,T_1868,2,+0.04053149 a_14^dagger a_13^dagger a_15 a_12 + +0.04053149 a_15^dagger a_12^dagger a_14 a_13
1869,T_1869,1,-0.50500065 a_15^dagger a_12^dagger a_15 a_12
1870,T_1870,1,-0.50500065 a_14^dagger a_13^dagger a_14 a_13
1871,T_1871,1,-0.46446915 a_15^dagger a_13^dagger a_15 a_13


In [2]:
# Cell 2: Build the H2 fermionic noncommutation graph

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from openfermion.transforms import normal_ordered

pd.set_option("display.max_colwidth", None)


def fermionic_commutator(A, B, tol=1e-12):
    """
    Compute [A, B] = AB - BA directly in the fermionic algebra.
    """
    C = normal_ordered(A * B - B * A)
    C.compress(abs_tol=tol)
    return C


def commute(A, B, tol=1e-12):
    """
    Return True if [A, B] = 0.
    """
    C = fermionic_commutator(A, B, tol=tol)
    return len(C.terms) == 0


# ------------------------------------------------------------
# Build noncommutation graph
# ------------------------------------------------------------
# Vertex i = Hermitian fermionic term T_i
# Edge (i, j) exists if [T_i, T_j] != 0

G = nx.Graph()

for i, T in enumerate(hermitian_terms):
    G.add_node(
        i,
        label=f"T_{i}",
        operator=T,
        operator_string=operator_to_string(T),
        number_of_monomials=len(T.terms),
    )

for i in range(len(hermitian_terms)):
    for j in range(i + 1, len(hermitian_terms)):
        Cij = fermionic_commutator(hermitian_terms[i], hermitian_terms[j])

        if len(Cij.terms) != 0:
            G.add_edge(
                i,
                j,
                commutator=Cij,
                commutator_string=operator_to_string(Cij),
            )


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

n_vertices = G.number_of_nodes()
n_total_pairs = n_vertices * (n_vertices - 1) // 2
n_noncommuting_pairs = G.number_of_edges()
n_commuting_pairs = n_total_pairs - n_noncommuting_pairs

print("=== Fermionic noncommutation graph summary ===")
print("Number of vertices / fermionic terms:", n_vertices)
print("Number of total unordered pairs:", n_total_pairs)
print("Number of noncommuting pairs / edges:", n_noncommuting_pairs)
print("Number of commuting pairs:", n_commuting_pairs)
print("Is graph bipartite?", nx.is_bipartite(G))


# ------------------------------------------------------------
# Vertex table
# ------------------------------------------------------------

vertex_rows = []

for i, data in G.nodes(data=True):
    vertex_rows.append(
        {
            "vertex": f"T_{i}",
            "degree": G.degree[i],
            "commutes_with_all": G.degree[i] == 0,
            "fermionic_term": data["operator_string"],
        }
    )

vertex_df = pd.DataFrame(vertex_rows)

print("\n=== Vertices: fermionic terms ===")
display(vertex_df)


# ------------------------------------------------------------
# Edge table
# ------------------------------------------------------------

edge_rows = []

for i, j, data in G.edges(data=True):
    edge_rows.append(
        {
            "source": f"T_{i}",
            "target": f"T_{j}",
            "meaning": f"[T_{i}, T_{j}] != 0",
            "commutator": data["commutator_string"],
        }
    )

edge_df = pd.DataFrame(edge_rows)

print("\n=== Edges: noncommuting pairs ===")
display(edge_df)


# # ------------------------------------------------------------
# # Draw graph
# # ------------------------------------------------------------

# plt.figure(figsize=(12, 8))

# pos = nx.kamada_kawai_layout(G)

# node_labels = {
#     i: f"T_{i}"
#     for i in G.nodes()
# }

# node_sizes = [
#     1000 + 250 * G.degree[i]
#     for i in G.nodes()
# ]

# nx.draw_networkx_nodes(G, pos, node_size=node_sizes)
# nx.draw_networkx_edges(G, pos, width=1.5)
# nx.draw_networkx_labels(G, pos, labels=node_labels, font_size=11, font_weight="bold")

# plt.title("H2 Fermionic Noncommutation Graph")
# plt.axis("off")
# plt.show()

=== Fermionic noncommutation graph summary ===
Number of vertices / fermionic terms: 1873
Number of total unordered pairs: 1753128
Number of noncommuting pairs / edges: 983776
Number of commuting pairs: 769352
Is graph bipartite? False

=== Vertices: fermionic terms ===


,vertex,degree,commutes_with_all,fermionic_term
0,T_0,0,True,+11.95858513 I
1,T_1,320,False,-25.74679421 a_0^dagger a_0
2,T_2,630,False,+0.44269974 a_0^dagger a_2 + +0.44269974 a_2^dagger a_0
3,T_3,630,False,-0.17053778 a_0^dagger a_8 + -0.17053778 a_8^dagger a_0
4,T_4,630,False,+0.35166139 a_0^dagger a_10 + +0.35166139 a_10^dagger a_0
...,...,...,...,...
1868,T_1868,1310,False,+0.04053149 a_14^dagger a_13^dagger a_15 a_12 + +0.04053149 a_15^dagger a_12^dagger a_14 a_13
1869,T_1869,743,False,-0.50500065 a_15^dagger a_12^dagger a_15 a_12
1870,T_1870,743,False,-0.50500065 a_14^dagger a_13^dagger a_14 a_13
1871,T_1871,709,False,-0.46446915 a_15^dagger a_13^dagger a_15 a_13



=== Edges: noncommuting pairs ===


,source,target,meaning,commutator
0,T_1,T_2,"[T_1, T_2] != 0",-11.39809899 a_0^dagger a_2 + +11.39809899 a_2^dagger a_0
1,T_1,T_3,"[T_1, T_3] != 0",+4.39080112 a_0^dagger a_8 + -4.39080112 a_8^dagger a_0
2,T_1,T_4,"[T_1, T_4] != 0",-9.05415335 a_0^dagger a_10 + +9.05415335 a_10^dagger a_0
3,T_1,T_38,"[T_1, T_38] != 0",+8.83882501 a_1^dagger a_0^dagger a_2 a_1 + -8.83882501 a_2^dagger a_1^dagger a_1 a_0
4,T_1,T_39,"[T_1, T_39] != 0",-3.53601260 a_1^dagger a_0^dagger a_8 a_1 + +3.53601260 a_8^dagger a_1^dagger a_1 a_0
...,...,...,...,...
983771,T_1864,T_1868,"[T_1864, T_1868] != 0",-0.01647157 a_14^dagger a_13^dagger a_11^dagger a_15 a_12 a_11 + +0.01647157 a_15^dagger a_12^dagger a_11^dagger a_14 a_13 a_11
983772,T_1865,T_1866,"[T_1865, T_1866] != 0",-0.02375404 a_13^dagger a_12^dagger a_15 a_14 + +0.02375404 a_15^dagger a_14^dagger a_13 a_12
983773,T_1866,T_1872,"[T_1866, T_1872] != 0",-0.02375404 a_13^dagger a_12^dagger a_15 a_14 + +0.02375404 a_15^dagger a_14^dagger a_13 a_12
983774,T_1868,T_1869,"[T_1868, T_1869] != 0",+0.02046843 a_14^dagger a_13^dagger a_15 a_12 + -0.02046843 a_15^dagger a_12^dagger a_14 a_13


In [3]:
# Cell 2 alpha: Faster H2 / molecular fermionic noncommutation graph
#
# Main idea:
#   1. Use cheap fermionic index rules first.
#   2. Only if rules cannot decide, use exact OpenFermion symbolic commutator.
#   3. Never build sparse/dense matrices.
#
# This cell assumes Cell 1 already defined:
#   hermitian_terms
#   operator_to_string

import time
from collections import Counter

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from openfermion.utils import commutator
from openfermion.transforms import normal_ordered

pd.set_option("display.max_colwidth", None)


# ------------------------------------------------------------
# Fermionic key utilities
# ------------------------------------------------------------

def key_modes(key):
    """
    Modes appearing in one OpenFermion monomial key.

    Example:
        ((3, 1), (0, 1), (3, 0), (0, 0)) -> {0, 3}
    """
    return frozenset(mode for mode, action in key)


def key_creations(key):
    """
    Creation modes in one monomial.
    """
    return frozenset(mode for mode, action in key if action == 1)


def key_annihilations(key):
    """
    Annihilation modes in one monomial.
    """
    return frozenset(mode for mode, action in key if action == 0)


def key_net_delta(key):
    """
    Net occupation change caused by a monomial.

    creation contributes +1
    annihilation contributes -1

    For example:
        a_2^dagger a_1^dagger a_3 a_0
    has delta:
        +1 on modes 2 and 1
        -1 on modes 3 and 0
    """
    delta = Counter()

    for mode, action in key:
        if action == 1:
            delta[mode] += 1
        else:
            delta[mode] -= 1

    return delta


def is_diagonal_key(key):
    """
    True if a monomial preserves occupation mode-by-mode.

    Examples:
        a_p^dagger a_p is diagonal.
        a_p^dagger a_q^dagger a_q a_p is diagonal.
        a_p^dagger a_q is not diagonal when p != q.
    """
    delta = key_net_delta(key)
    return all(value == 0 for value in delta.values())


def is_even_key(key):
    """
    Electronic Hamiltonian terms normally have even fermionic parity:
    length 0, 2, or 4.
    """
    return len(key) % 2 == 0


# ------------------------------------------------------------
# Safe monomial-level commutation rules
# ------------------------------------------------------------

def diagonal_key_commutes_with_key(diagonal_key, other_key):
    """
    Safe rule:

    A diagonal occupation operator depending on modes S commutes with another
    monomial if the other monomial has zero net occupation change on every
    mode in S.
    """
    support = key_modes(diagonal_key)
    delta = key_net_delta(other_key)

    return all(delta.get(mode, 0) == 0 for mode in support)


def no_cross_contractions_even_commute(key_a, key_b):
    """
    Safe rule for normal-ordered even fermionic monomials.

    If there are no possible cross contractions:
        annihilations(A) intersect creations(B) = empty
        annihilations(B) intersect creations(A) = empty

    then even monomials commute.

    This catches many cases beyond completely disjoint support.
    """
    if not is_even_key(key_a) or not is_even_key(key_b):
        return False

    a_ann = key_annihilations(key_a)
    a_cre = key_creations(key_a)

    b_ann = key_annihilations(key_b)
    b_cre = key_creations(key_b)

    return a_ann.isdisjoint(b_cre) and b_ann.isdisjoint(a_cre)


def monomial_pair_definitely_commutes(key_a, key_b):
    """
    Return (True, reason) only when we are sure two monomials commute.
    Return (False, None) if the rule cannot decide.

    Important:
        False here does NOT mean noncommuting.
        It only means "unknown; use exact symbolic fallback."
    """
    # Identity commutes with everything.
    if key_a == () or key_b == ():
        return True, "identity"

    # Any monomial commutes with itself.
    if key_a == key_b:
        return True, "same_monomial"

    # Diagonal occupation-like monomials commute with each other.
    if is_diagonal_key(key_a) and is_diagonal_key(key_b):
        return True, "diagonal_diagonal"

    # Diagonal with excitation-like term, if excitation preserves diagonal support.
    if is_diagonal_key(key_a) and diagonal_key_commutes_with_key(key_a, key_b):
        return True, "diagonal_support_preserved"

    if is_diagonal_key(key_b) and diagonal_key_commutes_with_key(key_b, key_a):
        return True, "diagonal_support_preserved"

    # Even monomials with no cross contractions commute.
    if no_cross_contractions_even_commute(key_a, key_b):
        return True, "no_cross_contractions_even"

    return False, None


# ------------------------------------------------------------
# Operator-level metadata and precheck
# ------------------------------------------------------------

def operator_metadata(op):
    """
    Precompute simple structural data for one FermionOperator.
    """
    keys = list(op.terms.keys())

    modes = set()
    for key in keys:
        modes.update(key_modes(key))

    return {
        "is_zero": len(keys) == 0,
        "only_identity": len(keys) == 1 and keys[0] == (),
        "modes": frozenset(modes),
        "is_even": all(is_even_key(key) for key in keys),
        "is_diagonal": all(is_diagonal_key(key) for key in keys),
        "number_of_monomials": len(keys),
    }


def operator_pair_definitely_commutes(A, B, meta_A, meta_B):
    """
    Return (True, reason) only for guaranteed-commuting pairs.
    Return (False, None) when unresolved.
    """
    if meta_A["is_zero"] or meta_B["is_zero"]:
        return True, "zero"

    if meta_A["only_identity"] or meta_B["only_identity"]:
        return True, "identity"

    # Very cheap global rule:
    # disjoint even fermionic operators commute.
    if (
        meta_A["is_even"]
        and meta_B["is_even"]
        and meta_A["modes"].isdisjoint(meta_B["modes"])
    ):
        return True, "disjoint_even_support"

    # Diagonal occupation-like operators commute with each other.
    if meta_A["is_diagonal"] and meta_B["is_diagonal"]:
        return True, "diagonal_diagonal"

    # More detailed but still cheap:
    # if every monomial pair has a safe commuting reason, the sums commute.
    reasons = Counter()

    for key_a in A.terms:
        for key_b in B.terms:
            ok, reason = monomial_pair_definitely_commutes(key_a, key_b)

            if not ok:
                return False, None

            reasons[reason] += 1

    if len(reasons) > 0:
        main_reason = reasons.most_common(1)[0][0]
        return True, f"all_monomial_pairs_{main_reason}"

    return False, None


# ------------------------------------------------------------
# Exact symbolic fallback
# ------------------------------------------------------------

def exact_symbolic_fermionic_commutator(A, B, tol=1e-12):
    """
    Exact symbolic commutator in fermionic algebra.

    This does not build a 2^n matrix.
    """
    C = normal_ordered(commutator(A, B))
    C.compress(abs_tol=tol)
    return C


# ------------------------------------------------------------
# Alpha graph builder
# ------------------------------------------------------------

def build_fermionic_noncommutation_graph_alpha(
    hermitian_terms,
    tol=1e-12,
    store_commutators=False,
):
    """
    Build noncommutation graph using:
        fast safe index rules first,
        exact symbolic OpenFermion fallback only when needed.

    Parameters
    ----------
    hermitian_terms:
        list of FermionOperator terms T_i from Cell 1.

    tol:
        numerical compression tolerance.

    store_commutators:
        False is recommended for large molecules.
        True is useful for H2 debugging, but can be memory-heavy.

    Returns
    -------
    G:
        networkx.Graph

    stats_df:
        pandas.DataFrame with timing and skip counts
    """
    t_start = time.perf_counter()

    G = nx.Graph()
    stats = Counter()

    metadata = [operator_metadata(T) for T in hermitian_terms]

    # Add vertices.
    for i, T in enumerate(hermitian_terms):
        G.add_node(
            i,
            label=f"T_{i}",
            operator=T,
            operator_string=operator_to_string(T),
            number_of_monomials=len(T.terms),
            modes=sorted(metadata[i]["modes"]),
            is_diagonal=metadata[i]["is_diagonal"],
            is_even=metadata[i]["is_even"],
        )

    n = len(hermitian_terms)

    # Pairwise graph construction.
    for i in range(n):
        A = hermitian_terms[i]
        meta_A = metadata[i]

        for j in range(i + 1, n):
            B = hermitian_terms[j]
            meta_B = metadata[j]

            stats["total_pairs"] += 1

            # 1. Fast guaranteed-commuting rules.
            definitely_commutes, reason = operator_pair_definitely_commutes(
                A, B, meta_A, meta_B
            )

            if definitely_commutes:
                stats["pairs_skipped_by_index_rules"] += 1
                stats[f"skip_{reason}"] += 1
                continue

            # 2. Exact symbolic fallback.
            stats["pairs_sent_to_exact_symbolic"] += 1

            Cij = exact_symbolic_fermionic_commutator(A, B, tol=tol)

            if len(Cij.terms) != 0:
                stats["noncommuting_edges"] += 1

                edge_data = {
                    "method": "exact_symbolic_fallback",
                    "meaning": f"[T_{i}, T_{j}] != 0",
                }

                if store_commutators:
                    edge_data["commutator"] = Cij
                    edge_data["commutator_string"] = operator_to_string(Cij)
                else:
                    edge_data["commutator_string"] = (
                        "(not stored; rerun with store_commutators=True)"
                    )

                G.add_edge(i, j, **edge_data)

            else:
                stats["exact_symbolic_found_commuting"] += 1

    elapsed = time.perf_counter() - t_start

    stats["vertices"] = n
    stats["edges"] = G.number_of_edges()
    stats["commuting_pairs"] = stats["total_pairs"] - G.number_of_edges()
    stats["elapsed_seconds"] = elapsed

    stats_df = pd.DataFrame([dict(stats)])

    return G, stats_df


# ------------------------------------------------------------
# Run alpha graph builder
# ------------------------------------------------------------

# For H2 debugging, you can set store_commutators=True.
# For larger molecules, keep this False.
G_alpha, stats_df = build_fermionic_noncommutation_graph_alpha(
    hermitian_terms,
    tol=1e-12,
    store_commutators=False,
)

print("=== Fermionic noncommutation graph alpha summary ===")
display(stats_df)

print("Number of vertices / fermionic terms:", G_alpha.number_of_nodes())
print("Number of noncommuting pairs / edges:", G_alpha.number_of_edges())
print("Is graph bipartite?", nx.is_bipartite(G_alpha))


# ------------------------------------------------------------
# Vertex table
# ------------------------------------------------------------

vertex_rows = []

for i, data in G_alpha.nodes(data=True):
    vertex_rows.append(
        {
            "vertex": f"T_{i}",
            "degree": G_alpha.degree[i],
            "commutes_with_all": G_alpha.degree[i] == 0,
            "number_of_monomials": data["number_of_monomials"],
            "modes": data["modes"],
            "is_diagonal": data["is_diagonal"],
            "fermionic_term": data["operator_string"],
        }
    )

vertex_df_alpha = pd.DataFrame(vertex_rows)

print("\n=== Alpha vertices: fermionic terms ===")
display(vertex_df_alpha)


# ------------------------------------------------------------
# Edge table
# ------------------------------------------------------------

edge_rows = []

for i, j, data in G_alpha.edges(data=True):
    edge_rows.append(
        {
            "source": f"T_{i}",
            "target": f"T_{j}",
            "meaning": data["meaning"],
            "method": data["method"],
            "commutator": data["commutator_string"],
        }
    )

edge_df_alpha = pd.DataFrame(edge_rows)

print("\n=== Alpha edges: noncommuting pairs ===")
display(edge_df_alpha)

# The commutation graph is too large for this.

# # ------------------------------------------------------------
# # Draw graph
# # ------------------------------------------------------------

# plt.figure(figsize=(12, 8))

# pos = nx.kamada_kawai_layout(G_alpha)

# node_labels = {
#     i: f"T_{i}"
#     for i in G_alpha.nodes()
# }

# node_sizes = [
#     1000 + 250 * G_alpha.degree[i]
#     for i in G_alpha.nodes()
# ]

# nx.draw_networkx_nodes(G_alpha, pos, node_size=node_sizes)
# nx.draw_networkx_edges(G_alpha, pos, width=1.5)
# nx.draw_networkx_labels(
#     G_alpha,
#     pos,
#     labels=node_labels,
#     font_size=11,
#     font_weight="bold",
# )

# plt.title("Fermionic Noncommutation Graph Alpha")
# plt.axis("off")
# plt.show()

=== Fermionic noncommutation graph alpha summary ===


,total_pairs,pairs_skipped_by_index_rules,skip_identity,pairs_sent_to_exact_symbolic,noncommuting_edges,skip_disjoint_even_support,skip_diagonal_diagonal,skip_all_monomial_pairs_diagonal_support_preserved,exact_symbolic_found_commuting,vertices,edges,commuting_pairs,elapsed_seconds
0,1753128,669360,1872,1083768,983776,658736,1920,6832,99992,1873,983776,769352,61.343122


Number of vertices / fermionic terms: 1873
Number of noncommuting pairs / edges: 983776
Is graph bipartite? False

=== Alpha vertices: fermionic terms ===


,vertex,degree,commutes_with_all,number_of_monomials,modes,is_diagonal,fermionic_term
0,T_0,0,True,1,[],True,+11.95858513 I
1,T_1,320,False,1,[0],True,-25.74679421 a_0^dagger a_0
2,T_2,630,False,2,"[0, 2]",False,+0.44269974 a_0^dagger a_2 + +0.44269974 a_2^dagger a_0
3,T_3,630,False,2,"[0, 8]",False,-0.17053778 a_0^dagger a_8 + -0.17053778 a_8^dagger a_0
4,T_4,630,False,2,"[0, 10]",False,+0.35166139 a_0^dagger a_10 + +0.35166139 a_10^dagger a_0
...,...,...,...,...,...,...,...
1868,T_1868,1310,False,2,"[12, 13, 14, 15]",False,+0.04053149 a_14^dagger a_13^dagger a_15 a_12 + +0.04053149 a_15^dagger a_12^dagger a_14 a_13
1869,T_1869,743,False,1,"[12, 15]",True,-0.50500065 a_15^dagger a_12^dagger a_15 a_12
1870,T_1870,743,False,1,"[13, 14]",True,-0.50500065 a_14^dagger a_13^dagger a_14 a_13
1871,T_1871,709,False,1,"[13, 15]",True,-0.46446915 a_15^dagger a_13^dagger a_15 a_13



=== Alpha edges: noncommuting pairs ===


,source,target,meaning,method,commutator
0,T_1,T_2,"[T_1, T_2] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
1,T_1,T_3,"[T_1, T_3] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
2,T_1,T_4,"[T_1, T_4] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
3,T_1,T_38,"[T_1, T_38] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
4,T_1,T_39,"[T_1, T_39] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
...,...,...,...,...,...
983771,T_1864,T_1868,"[T_1864, T_1868] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
983772,T_1865,T_1866,"[T_1865, T_1866] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
983773,T_1866,T_1872,"[T_1866, T_1872] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
983774,T_1868,T_1869,"[T_1868, T_1869] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)


In [4]:
print("Same edge set?")
print(set(G.edges()) == set(G_alpha.edges()))

Same edge set?
True


In [5]:
# Cell 3: Color graph, build commuting blocks, map JW/BK, verify commutation

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from itertools import combinations
from openfermion.ops import FermionOperator
from openfermion.transforms import normal_ordered, jordan_wigner, bravyi_kitaev

pd.set_option("display.max_colwidth", None)


# ------------------------------------------------------------
# 1. Color the noncommutation graph
# ------------------------------------------------------------
# Since edges mean noncommutation, each color class is a commuting group.

coloring = nx.coloring.greedy_color(G, strategy="largest_first")

color_groups = {}

for node, color in coloring.items():
    color_groups.setdefault(color, []).append(node)

for color in color_groups:
    color_groups[color] = sorted(color_groups[color])

color_names = [
    "red",
    "blue",
    "green",
    "orange",
    "purple",
    "brown",
    "pink",
    "gray",
]

color_name = {
    color: color_names[color] if color < len(color_names) else f"color_{color}"
    for color in color_groups
}

num_grouped_terms = sum(len(nodes) for nodes in color_groups.values())

print("Number of fermionic terms / vertices:", G.number_of_nodes())
print("Number of colors / commuting groups:", len(color_groups))
print("Number of grouped terms:", num_grouped_terms)

assert num_grouped_terms == G.number_of_nodes()


# ------------------------------------------------------------
# 2. Verify each color group is mutually commuting
# ------------------------------------------------------------

def verify_commuting_group(nodes, tol=1e-12):
    for i, j in combinations(nodes, 2):
        A = G.nodes[i]["operator"]
        B = G.nodes[j]["operator"]

        if not commute(A, B, tol=tol):
            return False

    return True


group_summary_rows = []

for color, nodes in sorted(color_groups.items()):
    group_summary_rows.append(
        {
            "color_id": color,
            "color_name": color_name[color],
            "number_of_terms": len(nodes),
            "vertices": [f"T_{i}" for i in nodes],
            "verified_mutually_commuting": verify_commuting_group(nodes),
        }
    )

group_summary_df = pd.DataFrame(group_summary_rows)

print("\n=== Commuting groups from graph coloring ===")
display(group_summary_df)


# ------------------------------------------------------------
# 3. Build Hamiltonian pieces by color
# ------------------------------------------------------------

H_by_color = {}

for color, nodes in sorted(color_groups.items()):
    H_color = FermionOperator.zero()

    for node in nodes:
        H_color += G.nodes[node]["operator"]

    H_color = normal_ordered(H_color)
    H_color.compress(abs_tol=1e-12)

    H_by_color[color] = H_color


color_block_rows = []

for color, nodes in sorted(color_groups.items()):
    for local_index, node in enumerate(nodes, start=1):
        color_block_rows.append(
            {
                "color_block": f"H_{color_name[color]}",
                "local_term_name": f"{color_name[color][0].upper()}_{local_index}",
                "vertex": f"T_{node}",
                "fermionic_term": G.nodes[node]["operator_string"],
            }
        )

color_block_df = pd.DataFrame(color_block_rows)

print("\n=== Which T_i belongs to which color block ===")
display(color_block_df)


print("\n=== Hamiltonian split by commuting color groups ===")

for color, H_color in H_by_color.items():
    nodes = color_groups[color]

    print("\n" + "=" * 80)
    print(f"H_{color_name[color]} consists of:")
    print(" + ".join([f"T_{node}" for node in nodes]))

    print(f"\nSummed operator H_{color_name[color]} =")
    print(H_color)


# ------------------------------------------------------------
# 4. Trotter ordering induced by color groups
# ------------------------------------------------------------

trotter_order = []

for color, nodes in sorted(color_groups.items()):
    for node in nodes:
        trotter_order.append(node)

print("\n=== Trotter order by commuting color groups ===")
print([f"T_{i}" for i in trotter_order])


# ------------------------------------------------------------
# 5. Helper functions for JW/BK output
# ------------------------------------------------------------

def sort_qubit_key(term):
    return (len(term), term)


def format_qubit_term(term):
    if term == ():
        return "I"

    return " ".join([f"{pauli}{qubit}" for qubit, pauli in term])


def qubit_coeff_to_str(c, digits=8):
    c = complex(c)
    if abs(c.imag) < 1e-12:
        return f"{c.real:+.{digits}f}"
    return f"{c.real:+.{digits}f}{c.imag:+.{digits}f}j"


def qubit_operator_to_string(op, digits=8):
    pieces = []

    for term, coeff in sorted(op.terms.items(), key=lambda item: sort_qubit_key(item[0])):
        pieces.append(f"{qubit_coeff_to_str(coeff, digits)} {format_qubit_term(term)}")

    if len(pieces) == 0:
        return "0"

    return " + ".join(pieces)


def apply_bk(op, n_qubits):
    try:
        return bravyi_kitaev(op, n_qubits=n_qubits)
    except TypeError:
        return bravyi_kitaev(op, n_qubits)


def qubit_commutator(A, B, tol=1e-12):
    C = A * B - B * A
    C.compress(abs_tol=tol)
    return C


def qubit_commute(A, B, tol=1e-12):
    C = qubit_commutator(A, B, tol=tol)
    return len(C.terms) == 0


n_qubits = molecule.n_qubits

print("\nNumber of qubits / spin orbitals:", n_qubits)


# ------------------------------------------------------------
# 6. Map each fermionic vertex T_i to JW(T_i) and BK(T_i)
# ------------------------------------------------------------

mapped_rows = []

for node in sorted(G.nodes()):
    T_i = G.nodes[node]["operator"]

    JW_T_i = jordan_wigner(T_i)
    JW_T_i.compress(abs_tol=1e-12)

    BK_T_i = apply_bk(T_i, n_qubits=n_qubits)
    BK_T_i.compress(abs_tol=1e-12)

    G.nodes[node]["JW_operator"] = JW_T_i
    G.nodes[node]["BK_operator"] = BK_T_i
    G.nodes[node]["JW_operator_string"] = qubit_operator_to_string(JW_T_i)
    G.nodes[node]["BK_operator_string"] = qubit_operator_to_string(BK_T_i)

    node_color = coloring[node]
    node_color_name = color_name[node_color]

    mapped_rows.append(
        {
            "color": node_color_name,
            "vertex": f"T_{node}",
            "fermionic_term": G.nodes[node]["operator_string"],
            "number_of_JW_Pauli_strings": len(JW_T_i.terms),
            "JW_transform": qubit_operator_to_string(JW_T_i),
            "number_of_BK_Pauli_strings": len(BK_T_i.terms),
            "BK_transform": qubit_operator_to_string(BK_T_i),
        }
    )

mapped_terms_df = pd.DataFrame(mapped_rows)

print("\n=== Fermionic terms mapped to JW and BK ===")
display(mapped_terms_df)


# ------------------------------------------------------------
# 7. Map each color block H_color to JW and BK
# ------------------------------------------------------------

JW_by_color = {}
BK_by_color = {}

block_rows = []

for color, H_color in sorted(H_by_color.items()):
    JW_color = jordan_wigner(H_color)
    JW_color.compress(abs_tol=1e-12)

    BK_color = apply_bk(H_color, n_qubits=n_qubits)
    BK_color.compress(abs_tol=1e-12)

    JW_by_color[color] = JW_color
    BK_by_color[color] = BK_color

    block_rows.append(
        {
            "color_block": f"H_{color_name[color]}",
            "fermionic_vertices": " + ".join([f"T_{node}" for node in color_groups[color]]),
            "number_of_fermionic_terms": len(color_groups[color]),
            "number_of_JW_Pauli_strings": len(JW_color.terms),
            "JW_block": qubit_operator_to_string(JW_color),
            "number_of_BK_Pauli_strings": len(BK_color.terms),
            "BK_block": qubit_operator_to_string(BK_color),
        }
    )

block_map_df = pd.DataFrame(block_rows)

print("\n=== Color blocks mapped to JW and BK ===")
display(block_map_df)


# ------------------------------------------------------------
# 8. Verify same-color terms commute after JW and BK
# ------------------------------------------------------------

verification_rows = []

for color, nodes in sorted(color_groups.items()):
    for i, j in combinations(nodes, 2):
        Ti = G.nodes[i]["operator"]
        Tj = G.nodes[j]["operator"]

        JW_Ti = G.nodes[i]["JW_operator"]
        JW_Tj = G.nodes[j]["JW_operator"]

        BK_Ti = G.nodes[i]["BK_operator"]
        BK_Tj = G.nodes[j]["BK_operator"]

        verification_rows.append(
            {
                "color_group": color_name[color],
                "pair": f"T_{i}, T_{j}",
                "fermionic_commute": commute(Ti, Tj),
                "JW_commute": qubit_commute(JW_Ti, JW_Tj),
                "BK_commute": qubit_commute(BK_Ti, BK_Tj),
            }
        )

verification_df = pd.DataFrame(verification_rows)

print("\n=== Verify commuting groups after JW/BK mapping ===")
display(verification_df)

Number of fermionic terms / vertices: 1873
Number of colors / commuting groups: 199
Number of grouped terms: 1873

=== Commuting groups from graph coloring ===


,color_id,color_name,number_of_terms,vertices,verified_mutually_commuting
0,0,red,13,"[T_0, T_42, T_74, T_75, T_419, T_1160, T_1194, T_1195, T_1411, T_1784, T_1788, T_1866, T_1868]",True
1,1,blue,19,"[T_28, T_29, T_32, T_102, T_132, T_180, T_622, T_663, T_672, T_702, T_754, T_865, T_1104, T_1406, T_1782, T_1787, T_1790, T_1817, T_1867]",True
2,2,green,17,"[T_27, T_30, T_31, T_333, T_334, T_377, T_431, T_457, T_476, T_693, T_768, T_853, T_1014, T_1189, T_1783, T_1814, T_1871]",True
3,3,orange,12,"[T_63, T_158, T_349, T_707, T_730, T_770, T_1102, T_1135, T_1781, T_1789, T_1813, T_1836]",True
4,4,purple,8,"[T_59, T_296, T_623, T_731, T_771, T_1103, T_1136, T_1144]",True
...,...,...,...,...,...
194,194,color_194,3,"[T_671, T_1113, T_1348]",True
195,195,color_195,5,"[T_215, T_713, T_976, T_1794, T_1798]",True
196,196,color_196,2,"[T_1228, T_1779]",True
197,197,color_197,3,"[T_668, T_802, T_1804]",True



=== Which T_i belongs to which color block ===


,color_block,local_term_name,vertex,fermionic_term
0,H_red,R_1,T_0,+11.95858513 I
1,H_red,R_2,T_42,-0.04537913 a_1^dagger a_0^dagger a_3 a_2 + -0.04537913 a_3^dagger a_2^dagger a_1 a_0
2,H_red,R_3,T_74,-0.00045357 a_2^dagger a_0^dagger a_10 a_8 + -0.00045357 a_10^dagger a_8^dagger a_2 a_0
3,H_red,R_4,T_75,+0.04537913 a_2^dagger a_1^dagger a_3 a_0 + +0.04537913 a_3^dagger a_0^dagger a_2 a_1
4,H_red,R_5,T_419,-0.00045357 a_3^dagger a_1^dagger a_11 a_9 + -0.00045357 a_11^dagger a_9^dagger a_3 a_1
...,...,...,...,...
1868,H_color_197,C_1,T_668,-0.00304060 a_14^dagger a_1^dagger a_14 a_9 + -0.00304060 a_14^dagger a_9^dagger a_14 a_1
1869,H_color_197,C_2,T_802,-0.03873187 a_8^dagger a_2^dagger a_10 a_2 + -0.03873187 a_10^dagger a_2^dagger a_8 a_2
1870,H_color_197,C_3,T_1804,-0.05776077 a_14^dagger a_8^dagger a_14 a_10 + -0.05776077 a_14^dagger a_10^dagger a_14 a_8
1871,H_color_198,C_1,T_222,-0.02771321 a_8^dagger a_0^dagger a_10 a_8 + -0.02771321 a_10^dagger a_8^dagger a_8 a_0



=== Hamiltonian split by commuting color groups ===

H_red consists of:
T_0 + T_42 + T_74 + T_75 + T_419 + T_1160 + T_1194 + T_1195 + T_1411 + T_1784 + T_1788 + T_1866 + T_1868

Summed operator H_red =
11.958585125793121 [] +
-0.045379131115623306 [1^ 0^ 3 2] +
-0.0004535682816878871 [2^ 0^ 10 8] +
0.045379131115623306 [2^ 1^ 3 0] +
0.045379131115623306 [3^ 0^ 2 1] +
-0.0004535682816878871 [3^ 1^ 11 9] +
-0.045379131115623306 [3^ 2^ 1 0] +
-0.04381070805019185 [5^ 4^ 7 6] +
0.03427143723234157 [6^ 4^ 14 12] +
0.04381070805019185 [6^ 5^ 7 4] +
0.04381070805019185 [7^ 4^ 6 5] +
0.03427143723234157 [7^ 5^ 15 13] +
-0.04381070805019185 [7^ 6^ 5 4] +
-0.0352391109362861 [9^ 8^ 11 10] +
-0.0004535682816878871 [10^ 8^ 2 0] +
0.0352391109362861 [10^ 9^ 11 8] +
0.0352391109362861 [11^ 8^ 10 9] +
-0.0004535682816878871 [11^ 9^ 3 1] +
-0.0352391109362861 [11^ 10^ 9 8] +
-0.040531494624598985 [13^ 12^ 15 14] +
0.03427143723234157 [14^ 12^ 6 4] +
0.040531494624598985 [14^ 13^ 15 12] +
0.0405314946

,color,vertex,fermionic_term,number_of_JW_Pauli_strings,JW_transform,number_of_BK_Pauli_strings,BK_transform
0,red,T_0,+11.95858513 I,1,+11.95858513 I,1,+11.95858513 I
1,color_55,T_1,-25.74679421 a_0^dagger a_0,2,-12.87339711 I + +12.87339711 Z0,2,-12.87339711 I + +12.87339711 Z0
2,color_41,T_2,+0.44269974 a_0^dagger a_2 + +0.44269974 a_2^dagger a_0,2,+0.22134987 X0 Z1 X2 + +0.22134987 Y0 Z1 Y2,2,+0.22134987 X0 Y1 Y2 + -0.22134987 Y0 Y1 X2
3,color_62,T_3,-0.17053778 a_0^dagger a_8 + -0.17053778 a_8^dagger a_0,2,-0.08526889 X0 Z1 Z2 Z3 Z4 Z5 Z6 Z7 X8 + -0.08526889 Y0 Z1 Z2 Z3 Z4 Z5 Z6 Z7 Y8,2,-0.08526889 X0 X1 X3 Y7 Y8 X9 X11 + +0.08526889 Y0 X1 X3 Y7 X8 X9 X11
4,color_53,T_4,+0.35166139 a_0^dagger a_10 + +0.35166139 a_10^dagger a_0,2,+0.17583069 X0 Z1 Z2 Z3 Z4 Z5 Z6 Z7 Z8 Z9 X10 + +0.17583069 Y0 Z1 Z2 Z3 Z4 Z5 Z6 Z7 Z8 Z9 Y10,2,+0.17583069 X0 X1 X3 Y7 Z9 Y10 X11 + -0.17583069 Y0 X1 X3 Y7 Z9 X10 X11
...,...,...,...,...,...,...,...
1868,red,T_1868,+0.04053149 a_14^dagger a_13^dagger a_15 a_12 + +0.04053149 a_15^dagger a_12^dagger a_14 a_13,8,-0.00506644 X12 X13 X14 X15 + -0.00506644 X12 X13 Y14 Y15 + -0.00506644 X12 Y13 X14 Y15 + +0.00506644 X12 Y13 Y14 X15 + +0.00506644 Y12 X13 X14 Y15 + -0.00506644 Y12 X13 Y14 X15 + -0.00506644 Y12 Y13 X14 X15 + -0.00506644 Y12 Y13 Y14 Y15,8,-0.00506644 X12 X14 + -0.00506644 Y12 Y14 + +0.00506644 X12 Z13 X14 + +0.00506644 Y12 Z13 Y14 + -0.00506644 Z7 Z11 X12 X14 Z15 + -0.00506644 Z7 Z11 Y12 Y14 Z15 + +0.00506644 Z7 Z11 X12 Z13 X14 Z15 + +0.00506644 Z7 Z11 Y12 Z13 Y14 Z15
1869,color_171,T_1869,-0.50500065 a_15^dagger a_12^dagger a_15 a_12,4,+0.12625016 I + -0.12625016 Z12 + -0.12625016 Z15 + +0.12625016 Z12 Z15,4,+0.12625016 I + -0.12625016 Z12 + -0.12625016 Z7 Z11 Z13 Z14 Z15 + +0.12625016 Z7 Z11 Z12 Z13 Z14 Z15
1870,color_170,T_1870,-0.50500065 a_14^dagger a_13^dagger a_14 a_13,4,+0.12625016 I + -0.12625016 Z13 + -0.12625016 Z14 + +0.12625016 Z13 Z14,4,+0.12625016 I + -0.12625016 Z14 + -0.12625016 Z12 Z13 + +0.12625016 Z12 Z13 Z14
1871,green,T_1871,-0.46446915 a_15^dagger a_13^dagger a_15 a_13,4,+0.11611729 I + -0.11611729 Z13 + -0.11611729 Z15 + +0.11611729 Z13 Z15,4,+0.11611729 I + -0.11611729 Z12 Z13 + +0.11611729 Z7 Z11 Z12 Z14 Z15 + -0.11611729 Z7 Z11 Z13 Z14 Z15



=== Color blocks mapped to JW and BK ===


,color_block,fermionic_vertices,number_of_fermionic_terms,number_of_JW_Pauli_strings,JW_block,number_of_BK_Pauli_strings,BK_block
0,H_red,T_0 + T_42 + T_74 + T_75 + T_419 + T_1160 + T_1194 + T_1195 + T_1411 + T_1784 + T_1788 + T_1866 + T_1868,13,49,+11.95858513 I + -0.01134478 X0 X1 Y2 Y3 + +0.01134478 X0 Y1 Y2 X3 + +0.01134478 Y0 X1 X2 Y3 + -0.01134478 Y0 Y1 X2 X3 + -0.01095268 X4 X5 Y6 Y7 + +0.01095268 X4 Y5 Y6 X7 + +0.01095268 Y4 X5 X6 Y7 + -0.01095268 Y4 Y5 X6 X7 + -0.00880978 X8 X9 Y10 Y11 + +0.00880978 X8 Y9 Y10 X11 + +0.00880978 Y8 X9 X10 Y11 + -0.00880978 Y8 Y9 X10 X11 + -0.01013287 X12 X13 Y14 Y15 + +0.01013287 X12 Y13 Y14 X15 + +0.01013287 Y12 X13 X14 Y15 + -0.01013287 Y12 Y13 X14 X15 + +0.00005670 X0 Z1 X2 X8 Z9 X10 + -0.00005670 X0 Z1 X2 Y8 Z9 Y10 + +0.00005670 X0 Z1 Y2 X8 Z9 Y10 + +0.00005670 X0 Z1 Y2 Y8 Z9 X10 + +0.00005670 Y0 Z1 X2 X8 Z9 Y10 + +0.00005670 Y0 Z1 X2 Y8 Z9 X10 + -0.00005670 Y0 Z1 Y2 X8 Z9 X10 + +0.00005670 Y0 Z1 Y2 Y8 Z9 Y10 + +0.00005670 X1 Z2 X3 X9 Z10 X11 + -0.00005670 X1 Z2 X3 Y9 Z10 Y11 + +0.00005670 X1 Z2 Y3 X9 Z10 Y11 + +0.00005670 X1 Z2 Y3 Y9 Z10 X11 + +0.00005670 Y1 Z2 X3 X9 Z10 Y11 + +0.00005670 Y1 Z2 X3 Y9 Z10 X11 + -0.00005670 Y1 Z2 Y3 X9 Z10 X11 + +0.00005670 Y1 Z2 Y3 Y9 Z10 Y11 + -0.00428393 X4 Z5 X6 X12 Z13 X14 + +0.00428393 X4 Z5 X6 Y12 Z13 Y14 + -0.00428393 X4 Z5 Y6 X12 Z13 Y14 + -0.00428393 X4 Z5 Y6 Y12 Z13 X14 + -0.00428393 Y4 Z5 X6 X12 Z13 Y14 + -0.00428393 Y4 Z5 X6 Y12 Z13 X14 + +0.00428393 Y4 Z5 Y6 X12 Z13 X14 + -0.00428393 Y4 Z5 Y6 Y12 Z13 Y14 + -0.00428393 X5 Z6 X7 X13 Z14 X15 + +0.00428393 X5 Z6 X7 Y13 Z14 Y15 + -0.00428393 X5 Z6 Y7 X13 Z14 Y15 + -0.00428393 X5 Z6 Y7 Y13 Z14 X15 + -0.00428393 Y5 Z6 X7 X13 Z14 Y15 + -0.00428393 Y5 Z6 X7 Y13 Z14 X15 + +0.00428393 Y5 Z6 Y7 X13 Z14 X15 + -0.00428393 Y5 Z6 Y7 Y13 Z14 Y15,49,+11.95858513 I + +0.01134478 X0 Z1 X2 + +0.01134478 Y0 Z1 Y2 + +0.01095268 X4 Z5 X6 + +0.01095268 Y4 Z5 Y6 + +0.00880978 X8 Z9 X10 + +0.00880978 Y8 Z9 Y10 + +0.01013287 X12 Z13 X14 + +0.01013287 Y12 Z13 Y14 + +0.01134478 X0 Z1 X2 Z3 + +0.01134478 Y0 Z1 Y2 Z3 + +0.00005670 X1 Z2 X9 Z10 + +0.00005670 Y1 Z3 Y9 Z11 + -0.00428393 X5 Z6 X13 Z14 + +0.00880978 X8 Z9 X10 Z11 + +0.00880978 Y8 Z9 Y10 Z11 + +0.00005670 Z0 X1 Z3 X9 Z10 + +0.00005670 Z0 Y1 Z2 Y9 Z11 + +0.00005670 X1 Z2 Z8 X9 Z11 + +0.00005670 Y1 Z3 Z8 Y9 Z10 + +0.01095268 Z3 X4 Z5 X6 Z7 + +0.01095268 Z3 Y4 Z5 Y6 Z7 + -0.00428393 Z3 Y5 Z11 Y13 Z15 + +0.00005670 X0 Y1 X2 X8 Y9 X10 + -0.00005670 X0 Y1 X2 Y8 Y9 Y10 + +0.00005670 X0 Y1 Y2 X8 Y9 Y10 + +0.00005670 X0 Y1 Y2 Y8 Y9 X10 + +0.00005670 Y0 Y1 X2 X8 Y9 Y10 + +0.00005670 Y0 Y1 X2 Y8 Y9 X10 + -0.00005670 Y0 Y1 Y2 X8 Y9 X10 + +0.00005670 Y0 Y1 Y2 Y8 Y9 Y10 + +0.00005670 Z0 X1 Z3 Z8 X9 Z11 + +0.00005670 Z0 Y1 Z2 Z8 Y9 Z10 + -0.00428393 Z3 Z4 X5 Z7 X13 Z14 + -0.00428393 Z3 Y5 Z7 Z12 Y13 Z14 + -0.00428393 X4 Y5 X6 X12 Y13 X14 + +0.00428393 X4 Y5 X6 Y12 Y13 Y14 + -0.00428393 X4 Y5 Y6 X12 Y13 Y14 + -0.00428393 X4 Y5 Y6 Y12 Y13 X14 + -0.00428393 Y4 Y5 X6 X12 Y13 Y14 + -0.00428393 Y4 Y5 X6 Y12 Y13 X14 + +0.00428393 Y4 Y5 Y6 X12 Y13 X14 + -0.00428393 Y4 Y5 Y6 Y12 Y13 Y14 + -0.00428393 Z4 Y5 Z6 Z12 Y13 Z14 + +0.01013287 Z7 Z11 X12 Z13 X14 Z15 + +0.01013287 Z7 Z11 Y12 Z13 Y14 Z15 + -0.00428393 Z3 Z4 X5 Z11 Z12 X13 Z15 + -0.00428393 Z4 Y5 Z6 Z7 Z11 Y13 Z15 + -0.00428393 X5 Z6 Z7 Z11 Z12 X13 Z15
1,H_blue,T_28 + T_29 + T_32 + T_102 + T_132 + T_180 + T_622 + T_663 + T_672 + T_702 + T_754 + T_865 + T_1104 + T_1406 + T_1782 + T_1787 + T_1790 + T_1817 + T_1867,19,43,-4.34789818 I + -0.17565851 Z0 + -0.19351217 Z1 + -0.10912438 Z2 + -0.12538899 Z3 + -0.17565851 Z4 + -0.11296968 Z5 + -0.10912438 Z6 + -0.11296968 Z7 + -0.11439536 Z8 + +2.98050665 Z9 + -0.11439536 Z10 + +2.20016254 Z11 + -0.11611729 Z12 + -0.12538899 Z13 + -0.11611729 Z14 + -0.19351217 Z15 + +0.17565851 Z0 Z4 + +0.19351217 Z1 Z15 + +0.10912438 Z2 Z6 + +0.12538899 Z3 Z13 + +0.11296968 Z5 Z7 + -0.01060820 X8 X10 + -0.01060820 Y8 Y10 + +0.11439536 Z8 Z10 + +0.11439536 Z9 Z11 + +0.11611729 Z12 Z14 + -0.21271281 X8 


=== Verify commuting groups after JW/BK mapping ===


,color_group,pair,fermionic_commute,JW_commute,BK_commute
0,red,"T_0, T_42",True,True,True
1,red,"T_0, T_74",True,True,True
2,red,"T_0, T_75",True,True,True
3,red,"T_0, T_419",True,True,True
4,red,"T_0, T_1160",True,True,True
...,...,...,...,...,...
8947,color_196,"T_1228, T_1779",True,True,True
8948,color_197,"T_668, T_802",True,True,True
8949,color_197,"T_668, T_1804",True,True,True
8950,color_197,"T_802, T_1804",True,True,True


In [6]:
# Find duplicated JW Pauli strings across fermionic terms T_i

from collections import defaultdict
import pandas as pd

pauli_usage = defaultdict(list)

for node in sorted(G.nodes()):
    JW_T = G.nodes[node]["JW_operator"]

    for pauli_key, coeff in JW_T.terms.items():
        pauli_string = format_qubit_term(pauli_key)

        pauli_usage[pauli_string].append(
            {
                "vertex": f"T_{node}",
                "coefficient": coeff,
                "fermionic_term": G.nodes[node]["operator_string"],
            }
        )

duplicate_rows = []

for pauli_string, appearances in pauli_usage.items():
    if len(appearances) > 1:
        duplicate_rows.append(
            {
                "JW_Pauli_string": pauli_string,
                "number_of_appearances": len(appearances),
                "appears_in_vertices": [x["vertex"] for x in appearances],
                "coefficients": [x["coefficient"] for x in appearances],
            }
        )

duplicate_jw_df = pd.DataFrame(duplicate_rows)
duplicate_jw_df = duplicate_jw_df.sort_values(
    "number_of_appearances",
    ascending=False
).reset_index(drop=True)

print("Total JW Pauli-string appearances:", sum(len(G.nodes[node]["JW_operator"].terms) for node in G.nodes()))
print("Number of unique JW Pauli strings:", len(pauli_usage))
print("Number of duplicated JW Pauli strings:", len(duplicate_jw_df))

display(duplicate_jw_df)

Total JW Pauli-string appearances: 12329
Number of unique JW Pauli strings: 5929
Number of duplicated JW Pauli strings: 4833


,JW_Pauli_string,number_of_appearances,appears_in_vertices,coefficients
0,I,137,"[T_0, T_1, T_5, T_9, T_12, T_15, T_18, T_21, T_24, T_27, T_29, T_31, T_32, T_33, T_34, T_35, T_36, T_37, T_65, T_78, T_102, T_124, T_158, T_184, T_214, T_235, T_249, T_272, T_282, T_322, T_334, T_378, T_386, T_410, T_423, T_457, T_480, T_510, T_535, T_549, T_570, T_580, T_611, T_623, T_664, T_672, T_686, T_707, T_723, T_754, T_774, T_801, T_816, T_828, T_844, T_853, T_885, T_896, T_932, T_939, T_970, T_987, T_1014, T_1033, T_1045, T_1060, T_1069, T_1093, T_1104, T_1137, T_1144, T_1155, T_1189, T_1200, T_1225, T_1243, T_1261, T_1280, T_1295, T_1327, T_1339, T_1375, T_1381, T_1406, T_1415, T_1433, T_1448, T_1463, T_1487, T_1499, T_1531, T_1537, T_1548, T_1573, T_1585, T_1601, T_1614, T_1627, T_1647, T_1657, ...]","[11.958585125793121, -12.873397105528989, -12.873397105528989, -3.21431432513072, -3.21431432513072, -2.788498991995936, -2.788498991995936, -2.788498991995935, -2.788498991995935, -3.094902010328627, -3.094902010328627, -2.314557905689917, -2.314557905689917, -2.469364464468404, -2.469364464468404, -2.469364464468406, -2.469364464468406, 1.031662472692932, 0.19849613291644927, 0.2098409156953551, 0.17565850546862888, 0.17800143599429125, 0.17565850546862877, 0.17800143599429114, 0.22842184036228022, 0.23480685318951328, 0.17222827004121635, 0.18183593826960326, 0.1935121738839605, 0.19853026824404124, 0.19351217388396072, 0.19853026824404146, 0.2098409156953551, 0.19849613291644927, 0.17800143599429125, 0.17565850546862888, 0.17800143599429114, 0.17565850546862877, 0.23480685318951328, 0.22842184036228022, 0.18183593826960326, 0.17222827004121635, 0.19853026824404124, 0.1935121738839605, 0.19853026824404146, 0.19351217388396072, 0.15315412960570418, 0.10912438299313622, 0.1403964753546454, 0.10912438299313622, 0.14039647535464533, 0.12471476761971263, 0.1492676869428829, 0.11201871698609418, 0.13550253109984928, 0.12538898897968145, 0.13847393612681186, 0.12538898897968154, 0.138473936126812, 0.1403964753546454, 0.10912438299313622, 0.14039647535464533, 0.10912438299313622, 0.1492676869428829, 0.12471476761971263, 0.13550253109984928, 0.11201871698609418, 0.13847393612681186, 0.12538898897968145, 0.138473936126812, 0.12538898897968154, 0.14582770935565428, 0.11296967831801041, 0.12392235533055837, 0.12884446815620498, 0.13628554125929992, 0.11273506231978409, 0.12637134040231102, 0.11522888382036564, 0.12472798894216569, 0.10894255103942999, 0.14174303197610932, 0.12392235533055837, 0.11296967831801041, 0.13628554125929992, 0.12884446815620498, 0.12637134040231102, 0.11273506231978409, 0.12472798894216569, 0.11522888382036564, 0.14174303197610932, 0.10894255103942999, 0.14582770935565426, 0.12884446815620498, 0.13628554125929992, 0.1127350623197841, 0.12637134040231118, 0.10894255103942994, 0.14174303197610916, 0.11522888382036565, ...]"
1,Z10,16,"[T_31, T_249, T_570, T_828, T_1060, T_1261, T_1448, T_1601, T_1711, T_1787, T_1813, T_1836, T_1839, T_1841, T_1846, T_1850]","[2.314557905689917, -0.17222827004121635, -0.18183593826960326, -0.11201871698609418, -0.13550253109984928, -0.11273506231978409, -0.12637134040231102, -0.1127350623197841, -0.12637134040231118, -0.11439536123713774, -0.12320513897120927, -0.13228582460883975, -0.10159735824737096, -0.1268539688386498, -0.10159735824737101, -0.12685396883864966]"
2,Z6,16,"[T_21, T_158, T_480, T_754, T_987, T_1189, T_1381, T_1548, T_1573, T_1585, T_1601, T_1614, T_1627, T_1647, T_1657, T_1680]","[2.788498991995935, -0.17565850546862877, -0.17800143599429114, -0.10912438299313622, -0.14039647535464533, -0.11296967831801041, -0.12392235533055837, -0.14582770935565426, -0.12884446815620498, -0.13628554125929992, -0.1127350623197841, -0.12637134040231118, -0.10894255103942994, -0.14174303197610916, -0.11522888382036565, -0.12472798894216572]"
3,Z4,16,"[T_15, T_102, T_423, T_707, T_939, T_1155, T_1189, T_1200, T_1225, T_1243, T_1261, T_1280, T_1295, T_1327, T_1339, T_1375]","[2.788498991995936,

In [7]:
# Compute Pauli-duplication ratio for JW and BK

from collections import defaultdict
import pandas as pd

from openfermion.ops import FermionOperator
from openfermion.transforms import jordan_wigner, bravyi_kitaev, normal_ordered


def pauli_support(qubit_op, tol=1e-12, include_identity=True):
    """
    Return the set of Pauli strings with nonzero coefficients.
    """
    qubit_op.compress(abs_tol=tol)

    support = set()

    for pauli_key, coeff in qubit_op.terms.items():
        if abs(coeff) <= tol:
            continue

        if not include_identity and pauli_key == ():
            continue

        support.add(pauli_key)

    return support


def map_fermion_to_qubit(op, mapping="JW", n_qubits=None):
    """
    Map a FermionOperator to a QubitOperator using JW or BK.
    """
    mapping = mapping.upper()

    if mapping == "JW":
        qop = jordan_wigner(op)

    elif mapping == "BK":
        if n_qubits is None:
            raise ValueError("n_qubits is required for BK.")

        try:
            qop = bravyi_kitaev(op, n_qubits=n_qubits)
        except TypeError:
            qop = bravyi_kitaev(op, n_qubits)

    else:
        raise ValueError("mapping must be 'JW' or 'BK'.")

    qop.compress(abs_tol=1e-12)
    return qop


def pauli_duplication_ratio(
    fermionic_terms,
    mapping="JW",
    n_qubits=None,
    include_identity=True,
    tol=1e-12,
):
    """
    Compute

        sum_alpha #mapping(H_alpha) / #mapping(H)

    where H = sum_alpha H_alpha.

    Also returns a duplicate-use table.
    """

    H_full = FermionOperator.zero()

    numerator = 0
    union_support = set()
    pauli_usage = defaultdict(list)

    for alpha, H_alpha in enumerate(fermionic_terms):
        H_full += H_alpha

        Q_alpha = map_fermion_to_qubit(
            H_alpha,
            mapping=mapping,
            n_qubits=n_qubits,
        )

        support_alpha = pauli_support(
            Q_alpha,
            tol=tol,
            include_identity=include_identity,
        )

        numerator += len(support_alpha)
        union_support |= support_alpha

        for pauli_key, coeff in Q_alpha.terms.items():
            if abs(coeff) <= tol:
                continue

            if not include_identity and pauli_key == ():
                continue

            pauli_usage[pauli_key].append(
                {
                    "vertex": f"T_{alpha}",
                    "coefficient": coeff,
                }
            )

    H_full = normal_ordered(H_full)
    H_full.compress(abs_tol=tol)

    Q_full = map_fermion_to_qubit(
        H_full,
        mapping=mapping,
        n_qubits=n_qubits,
    )

    full_support = pauli_support(
        Q_full,
        tol=tol,
        include_identity=include_identity,
    )

    denominator = len(full_support)

    duplication_ratio = numerator / denominator
    union_ratio = numerator / len(union_support)

    summary_df = pd.DataFrame(
        [
            {
                "mapping": mapping.upper(),
                "include_identity": include_identity,
                "sum_alpha_number_of_Pauli_strings": numerator,
                "number_of_unique_Pauli_strings_before_cancellation": len(union_support),
                "number_of_Pauli_strings_in_full_H": denominator,
                "duplication_ratio": duplication_ratio,
                "raw_reuse_ratio_before_cancellation": union_ratio,
            }
        ]
    )

    duplicate_rows = []

    for pauli_key, appearances in pauli_usage.items():
        if len(appearances) <= 1:
            continue

        combined_coeff = Q_full.terms.get(pauli_key, 0.0)

        duplicate_rows.append(
            {
                "Pauli_string": format_qubit_term(pauli_key),
                "number_of_appearances": len(appearances),
                "appears_in_vertices": [x["vertex"] for x in appearances],
                "individual_coefficients": [complex(x["coefficient"]) for x in appearances],
                "combined_coefficient_in_full_H": complex(combined_coeff),
                "survives_in_full_H": abs(combined_coeff) > tol,
            }
        )

    duplicate_df = pd.DataFrame(duplicate_rows)

    if len(duplicate_df) > 0:
        duplicate_df = duplicate_df.sort_values(
            "number_of_appearances",
            ascending=False,
        ).reset_index(drop=True)

    return summary_df, duplicate_df


# ------------------------------------------------------------
# Run for JW
# ------------------------------------------------------------

n_qubits = molecule.n_qubits

jw_summary_df, jw_duplicate_df = pauli_duplication_ratio(
    hermitian_terms,
    mapping="JW",
    n_qubits=n_qubits,
    include_identity=True,
)

print("=== JW Pauli-duplication ratio ===")
display(jw_summary_df)

print("\n=== Duplicated JW Pauli strings ===")
display(jw_duplicate_df)


# ------------------------------------------------------------
# Optional: run for BK too
# ------------------------------------------------------------

bk_summary_df, bk_duplicate_df = pauli_duplication_ratio(
    hermitian_terms,
    mapping="BK",
    n_qubits=n_qubits,
    include_identity=True,
)

print("=== BK Pauli-duplication ratio ===")
display(bk_summary_df)

print("\n=== Duplicated BK Pauli strings ===")
display(bk_duplicate_df)

=== JW Pauli-duplication ratio ===


,mapping,include_identity,sum_alpha_number_of_Pauli_strings,number_of_unique_Pauli_strings_before_cancellation,number_of_Pauli_strings_in_full_H,duplication_ratio,raw_reuse_ratio_before_cancellation
0,JW,True,12329,5929,3609,3.416182,2.07944



=== Duplicated JW Pauli strings ===


,Pauli_string,number_of_appearances,appears_in_vertices,individual_coefficients,combined_coefficient_in_full_H,survives_in_full_H
0,I,137,"[T_0, T_1, T_5, T_9, T_12, T_15, T_18, T_21, T_24, T_27, T_29, T_31, T_32, T_33, T_34, T_35, T_36, T_37, T_65, T_78, T_102, T_124, T_158, T_184, T_214, T_235, T_249, T_272, T_282, T_322, T_334, T_378, T_386, T_410, T_423, T_457, T_480, T_510, T_535, T_549, T_570, T_580, T_611, T_623, T_664, T_672, T_686, T_707, T_723, T_754, T_774, T_801, T_816, T_828, T_844, T_853, T_885, T_896, T_932, T_939, T_970, T_987, T_1014, T_1033, T_1045, T_1060, T_1069, T_1093, T_1104, T_1137, T_1144, T_1155, T_1189, T_1200, T_1225, T_1243, T_1261, T_1280, T_1295, T_1327, T_1339, T_1375, T_1381, T_1406, T_1415, T_1433, T_1448, T_1463, T_1487, T_1499, T_1531, T_1537, T_1548, T_1573, T_1585, T_1601, T_1614, T_1627, T_1647, T_1657, ...]","[(11.958585125793121+0j), (-12.873397105528989+0j), (-12.873397105528989+0j), (-3.21431432513072+0j), (-3.21431432513072+0j), (-2.788498991995936+0j), (-2.788498991995936+0j), (-2.788498991995935+0j), (-2.788498991995935+0j), (-3.094902010328627+0j), (-3.094902010328627+0j), (-2.314557905689917+0j), (-2.314557905689917+0j), (-2.469364464468404+0j), (-2.469364464468404+0j), (-2.469364464468406+0j), (-2.469364464468406+0j), (1.031662472692932+0j), (0.19849613291644927+0j), (0.2098409156953551+0j), (0.17565850546862888+0j), (0.17800143599429125+0j), (0.17565850546862877+0j), (0.17800143599429114+0j), (0.22842184036228022+0j), (0.23480685318951328+0j), (0.17222827004121635+0j), (0.18183593826960326+0j), (0.1935121738839605+0j), (0.19853026824404124+0j), (0.19351217388396072+0j), (0.19853026824404146+0j), (0.2098409156953551+0j), (0.19849613291644927+0j), (0.17800143599429125+0j), (0.17565850546862888+0j), (0.17800143599429114+0j), (0.17565850546862877+0j), (0.23480685318951328+0j), (0.22842184036228022+0j), (0.18183593826960326+0j), (0.17222827004121635+0j), (0.19853026824404124+0j), (0.1935121738839605+0j), (0.19853026824404146+0j), (0.19351217388396072+0j), (0.15315412960570418+0j), (0.10912438299313622+0j), (0.1403964753546454+0j), (0.10912438299313622+0j), (0.14039647535464533+0j), (0.12471476761971263+0j), (0.1492676869428829+0j), (0.11201871698609418+0j), (0.13550253109984928+0j), (0.12538898897968145+0j), (0.13847393612681186+0j), (0.12538898897968154+0j), (0.138473936126812+0j), (0.1403964753546454+0j), (0.10912438299313622+0j), (0.14039647535464533+0j), (0.10912438299313622+0j), (0.1492676869428829+0j), (0.12471476761971263+0j), (0.13550253109984928+0j), (0.11201871698609418+0j), (0.13847393612681186+0j), (0.12538898897968145+0j), (0.138473936126812+0j), (0.12538898897968154+0j), (0.14582770935565428+0j), (0.11296967831801041+0j), (0.12392235533055837+0j), (0.12884446815620498+0j), (0.13628554125929992+0j), (0.11273506231978409+0j), (0.12637134040231102+0j), (0.11522888382036564+0j), (0.12472798894216569+0j), (0.10894255103942999+0j), (0.14174303197610932+0j), (0.12392235533055837+0j), (0.11296967831801041+0j), (0.13628554125929992+0j), (0.12884446815620498+0j), (0.12637134040231102+0j), (0.11273506231978409+0j), (0.12472798894216569+0j), (0.11522888382036564+0j), (0.14174303197610932+0j), (0.10894255103942999+0j), (0.14582770935565426+0j), (0.12884446815620498+0j), (0.13628554125929992+0j), (0.1127350623197841+0j), (0.12637134040231118+0j), (0.10894255103942994+0j), (0.14174303197610916+0j), (0.11522888382036565+0j), ...]",-34.032200+ 0.000000j,True
1,Z10,16,"[T_31, T_249, T_570, T_828, T_1060, T_1261, T_1448, T_1601, T_1711, T_1787, T_1813, T_1836, T_1839, T_1841, T_1846, T_1850]","[(2.314557905689917+0j), (-0.17222827004121635+0j), (-0.18183593826960326+0j), (-0.11201871698609418+0j), (-0.13550253109984928+0j), (-0.11273506231978409+0j), (-0.12637134040231102+0j), (-0.1127350623197841+0j), (-0.12637134040231118+0j), (-0.11439536123713774+0j), (-0.12320513897120927+0j), (-0.13228582460883975+0j), (-0.10159735824737096+0j), (-0.1268539688386498+0j), (-0.10159735824737101+0j), 

=== BK Pauli-duplication ratio ===


,mapping,include_identity,sum_alpha_number_of_Pauli_strings,number_of_unique_Pauli_strings_before_cancellation,number_of_Pauli_strings_in_full_H,duplication_ratio,raw_reuse_ratio_before_cancellation
0,BK,True,12329,5929,3609,3.416182,2.07944



=== Duplicated BK Pauli strings ===


,Pauli_string,number_of_appearances,appears_in_vertices,individual_coefficients,combined_coefficient_in_full_H,survives_in_full_H
0,I,137,"[T_0, T_1, T_5, T_9, T_12, T_15, T_18, T_21, T_24, T_27, T_29, T_31, T_32, T_33, T_34, T_35, T_36, T_37, T_65, T_78, T_102, T_124, T_158, T_184, T_214, T_235, T_249, T_272, T_282, T_322, T_334, T_378, T_386, T_410, T_423, T_457, T_480, T_510, T_535, T_549, T_570, T_580, T_611, T_623, T_664, T_672, T_686, T_707, T_723, T_754, T_774, T_801, T_816, T_828, T_844, T_853, T_885, T_896, T_932, T_939, T_970, T_987, T_1014, T_1033, T_1045, T_1060, T_1069, T_1093, T_1104, T_1137, T_1144, T_1155, T_1189, T_1200, T_1225, T_1243, T_1261, T_1280, T_1295, T_1327, T_1339, T_1375, T_1381, T_1406, T_1415, T_1433, T_1448, T_1463, T_1487, T_1499, T_1531, T_1537, T_1548, T_1573, T_1585, T_1601, T_1614, T_1627, T_1647, T_1657, ...]","[(11.958585125793121+0j), (-12.873397105528989+0j), (-12.873397105528989+0j), (-3.21431432513072+0j), (-3.21431432513072+0j), (-2.788498991995936+0j), (-2.788498991995936+0j), (-2.788498991995935+0j), (-2.788498991995935+0j), (-3.094902010328627+0j), (-3.094902010328627+0j), (-2.314557905689917+0j), (-2.314557905689917+0j), (-2.469364464468404+0j), (-2.469364464468404+0j), (-2.469364464468406+0j), (-2.469364464468406+0j), (1.031662472692932+0j), (0.19849613291644927+0j), (0.2098409156953551+0j), (0.17565850546862888+0j), (0.17800143599429125+0j), (0.17565850546862877+0j), (0.17800143599429114+0j), (0.22842184036228022+0j), (0.23480685318951328+0j), (0.17222827004121635+0j), (0.18183593826960326+0j), (0.1935121738839605+0j), (0.19853026824404124+0j), (0.19351217388396072+0j), (0.19853026824404146+0j), (0.2098409156953551+0j), (0.19849613291644927+0j), (0.17800143599429125+0j), (0.17565850546862888+0j), (0.17800143599429114+0j), (0.17565850546862877+0j), (0.23480685318951328+0j), (0.22842184036228022+0j), (0.18183593826960326+0j), (0.17222827004121635+0j), (0.19853026824404124+0j), (0.1935121738839605+0j), (0.19853026824404146+0j), (0.19351217388396072+0j), (0.15315412960570418+0j), (0.10912438299313622+0j), (0.1403964753546454+0j), (0.10912438299313622+0j), (0.14039647535464533+0j), (0.12471476761971263+0j), (0.1492676869428829+0j), (0.11201871698609418+0j), (0.13550253109984928+0j), (0.12538898897968145+0j), (0.13847393612681186+0j), (0.12538898897968154+0j), (0.138473936126812+0j), (0.1403964753546454+0j), (0.10912438299313622+0j), (0.14039647535464533+0j), (0.10912438299313622+0j), (0.1492676869428829+0j), (0.12471476761971263+0j), (0.13550253109984928+0j), (0.11201871698609418+0j), (0.13847393612681186+0j), (0.12538898897968145+0j), (0.138473936126812+0j), (0.12538898897968154+0j), (0.14582770935565428+0j), (0.11296967831801041+0j), (0.12392235533055837+0j), (0.12884446815620498+0j), (0.13628554125929992+0j), (0.11273506231978409+0j), (0.12637134040231102+0j), (0.11522888382036564+0j), (0.12472798894216569+0j), (0.10894255103942999+0j), (0.14174303197610932+0j), (0.12392235533055837+0j), (0.11296967831801041+0j), (0.13628554125929992+0j), (0.12884446815620498+0j), (0.12637134040231102+0j), (0.11273506231978409+0j), (0.12472798894216569+0j), (0.11522888382036564+0j), (0.14174303197610932+0j), (0.10894255103942999+0j), (0.14582770935565426+0j), (0.12884446815620498+0j), (0.13628554125929992+0j), (0.1127350623197841+0j), (0.12637134040231118+0j), (0.10894255103942994+0j), (0.14174303197610916+0j), (0.11522888382036565+0j), ...]",-34.032200+ 0.000000j,True
1,Z10,16,"[T_31, T_249, T_570, T_828, T_1060, T_1261, T_1448, T_1601, T_1711, T_1787, T_1813, T_1836, T_1839, T_1841, T_1846, T_1850]","[(2.314557905689917+0j), (-0.17222827004121635+0j), (-0.18183593826960326+0j), (-0.11201871698609418+0j), (-0.13550253109984928+0j), (-0.11273506231978409+0j), (-0.12637134040231102+0j), (-0.1127350623197841+0j), (-0.12637134040231118+0j), (-0.11439536123713774+0j), (-0.12320513897120927+0j), (-0.13228582460883975+0j), (-0.10159735824737096+0j), (-0.1268539688386498+0j), (-0.10159735824737101+0j), 